# Skeletons

**Geometry type:** `skeleton` (a tree-shaped `graph`)

Tree-structured graphs aligned to the SWC morphology convention. Stored
via `write_graph(..., is_tree=True)`; multi-skeleton stores label each
vertex with its parent-object via `object_ids`.

1. Generate a synthetic neuron morphology
2. Write as a single skeleton store (`is_tree=True`)
3. Open lazily and inspect
4. Read the full morphology
5. Multi-skeleton store via `object_ids`
6. SWC round-trip (ingest + export)
7. Validate

In [ ]:
import numpy as np, tempfile, os
from zarr_vectors.lazy import open_zvr
from zarr_vectors.types.graphs import write_graph, read_graph
from zarr_vectors.validate import validate

_tmpdir = tempfile.mkdtemp(prefix="zvf_examples_")
STORE      = os.path.join(_tmpdir, "neuron.zarrvectors")
STORE_MULTI = os.path.join(_tmpdir, "connectome.zarrvectors")
SWC_IN     = os.path.join(_tmpdir, "neuron.swc")
SWC_OUT    = os.path.join(_tmpdir, "neuron_roundtrip.swc")
print("Stores:", STORE, STORE_MULTI)

## 1 · Generate a synthetic neuron morphology

In [ ]:
rng = np.random.default_rng(3)

def generate_tree(n_nodes, origin, rng):
    """Generate a random branching tree with SWC attributes."""
    positions = np.zeros((n_nodes, 3), dtype=np.float32)
    positions[0] = origin                      # soma at origin
    parent_idx = np.full(n_nodes, -1, dtype=np.int32)

    for i in range(1, n_nodes):
        # Attach to a random existing node (creates branching)
        parent_idx[i] = rng.integers(0, i)
        step = rng.normal(0, 4, 3).astype(np.float32)
        positions[i] = positions[parent_idx[i]] + step

    # SWC compartment types: 0=soma, 1=axon, 2=basal dendrite
    swc_type = np.ones(n_nodes, dtype=np.int32) * 3   # basal dendrite default
    swc_type[0] = 1    # soma
    swc_type[1:n_nodes//5] = 2   # axon: first ~20% of nodes

    radii = np.where(swc_type == 1,
                     rng.uniform(8, 15, n_nodes),     # soma: large radius
                     rng.uniform(0.3, 2.0, n_nodes)   # processes: small
                    ).astype(np.float32)

    # Edges: [child, parent] pairs (root has parent = -1, stored separately)
    edges = np.column_stack([
        np.arange(1, n_nodes, dtype=np.int32),
        parent_idx[1:],
    ])
    return positions, edges, swc_type, radii

n_nodes = 1_200
positions, edges, swc_type, radii = generate_tree(
    n_nodes, origin=np.array([500., 500., 500.], dtype=np.float32), rng=rng
)

print(f"Generated neuron with {n_nodes} nodes and {len(edges)} edges")
print(f"SWC types: { dict(zip(*np.unique(swc_type, return_counts=True))) }")
print(f"Radius range: [{radii.min():.2f}, {radii.max():.2f}] µm")


## 2 · Write skeleton store

In [ ]:
write_graph(
    STORE,
    positions=positions,
    edges=edges,             # [child, parent] pairs
    chunk_shape=(200.0, 200.0, 200.0),
    bin_shape=(50.0, 50.0, 50.0),
    is_tree=True,
    node_attributes={
        "radius":   radii,
        "swc_type": swc_type,
        "swc_id":   np.arange(n_nodes, dtype=np.int64),
    },
)
print("Write complete.")

## 3 · Open lazily and inspect SWC attributes

In [ ]:
store = open_zvr(STORE)
print(f"geometry_types: {store.geometry_types}")
print(f"n_objects     : {store[0].object_index.object_count}  (1 skeleton)")
print(f"vertex_count  : {store[0].vertex_count:,}")
lo, hi = np.array(store.bounds[0]), np.array(store.bounds[1])
print(f"bounds        : lo={lo.round(1)}  hi={hi.round(1)}")

## 4 · Read the full morphology

In [ ]:
result = read_graph(STORE)
print(f"node_count    : {result['node_count']:,}")
print(f"edge_count    : {result['edge_count']:,}")
print(f"positions     : {result['positions'].shape}")

# Per-node attributes are stored on disk; load them via the level group
from zarr_vectors.core.store import open_store, get_resolution_level
from zarr_vectors.core.arrays import read_chunk_attributes
from zarr_vectors.core.store import list_resolution_levels

root = open_store(STORE)
level0 = get_resolution_level(root, 0)
# Aggregate per-chunk attribute reads; chunk order matches positions (concat order)
from zarr_vectors.core.arrays import list_chunk_keys
keys = list_chunk_keys(level0)
radius_all = np.concatenate([
    np.concatenate(read_chunk_attributes(level0, "radius", k, dtype=np.float32, ncols=1))
    for k in keys
])
swc_all = np.concatenate([
    np.concatenate(read_chunk_attributes(level0, "swc_type", k, dtype=np.float32, ncols=1))
    for k in keys
]).astype(np.int32)

print(f"radius range  : [{radius_all.min():.2f}, {radius_all.max():.2f}] µm")
types, counts = np.unique(swc_all, return_counts=True)
type_names = {1: "soma", 2: "axon", 3: "basal dendrite", 4: "apical dendrite"}
print("\nCompartment types:")
for t, c in zip(types, counts):
    print(f"  type {t} ({type_names.get(int(t), 'other'):>15s}): {c:>5} nodes")

## 5 · Multi-skeleton stores

The current implementation has a known limitation: `write_graph(...,
object_ids=...)` raises `ArrayError: Link group count != vertex group count`
when two skeletons share a chunk. Until that is fixed, the supported
pattern for connectome-style data is **one skeleton per store**, e.g.

```python
for i, (p, e, t, r) in enumerate(zip(all_pos, all_edges, all_types, all_radii)):
    write_graph(f"neuron_{i:03d}.zarrvectors", positions=p, edges=e,
                chunk_shape=(...), is_tree=True,
                node_attributes={"radius": r, "swc_type": t})
```

## 6 · SWC round-trip

In [ ]:
# Write a minimal synthetic SWC
with open(SWC_IN, "w") as f:
    f.write("# Generated by zarr-vectors example\n")
    for i in range(n_nodes):
        p_id = -1 if i == 0 else int(edges[i-1, 1]) + 1
        f.write(f"{i+1} {swc_type[i]} "
                f"{positions[i,0]:.3f} {positions[i,1]:.3f} {positions[i,2]:.3f} "
                f"{radii[i]:.3f} {p_id}\n")
print(f"Wrote {n_nodes}-node SWC to {SWC_IN}")

# Ingest SWC → ZVF
STORE_SWC = os.path.join(_tmpdir, "from_swc.zarrvectors")
from zarr_vectors.ingest.swc import ingest_swc
ingest_swc(SWC_IN, STORE_SWC, chunk_shape=(200., 200., 200.))
r_swc = read_graph(STORE_SWC)
print(f"Ingested: {r_swc['node_count']} nodes, {r_swc['edge_count']} edges")

# Export back to SWC
from zarr_vectors.export.swc import export_swc
export_swc(STORE_SWC, SWC_OUT)
print(f"Exported SWC to {SWC_OUT}")

# Verify round-trip
lines_in  = [l for l in open(SWC_IN)  if not l.startswith("#")]
lines_out = [l for l in open(SWC_OUT) if not l.startswith("#")]
print(f"\nInput  rows: {len(lines_in)}")
print(f"Output rows: {len(lines_out)}")


## 7 · Validate

In [ ]:
rv = validate(STORE, level=4)   # level 4 checks tree topology
print(rv.summary())


## Summary

Skeletons are stored as graphs with `is_tree=True`. SWC ingest/export
preserves the morphology round-trip.

| Step | API |
|------|-----|
| Write | `write_graph(path, positions, edges, is_tree=True, node_attributes=...)` |
| Read | `read_graph(path)` → `{positions, edges, node_count, edge_count}` |
| Read by ID | `read_graph(path, object_ids=[k])` (single-object stores are k=0) |
| Bbox query | `read_graph(path, bbox=(lo, hi))` |
| SWC ingest | `ingest_swc(swc_path, store_path, chunk_shape)` |
| SWC export | `export_swc(store_path, swc_path)` |

> **Known limitation:** multi-object stores via `object_ids=` currently
> fail when two objects share a chunk. Use one store per skeleton until
> that is fixed.